In [1]:
import torch
from pathlib import Path

ROOT = Path(r"D:\PS07_CT2MRI")
CHECKPOINT_PATH = ROOT / "models" / "latest_checkpoint.pth"

print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("Checkpoint size (MB):",
      round(CHECKPOINT_PATH.stat().st_size / (1024 * 1024), 2))

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", checkpoint.keys())

Checkpoint exists: True
Checkpoint size (MB): 23.39

Checkpoint type: <class 'dict'>
Checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'train_loss'])


In [2]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import torch

from skimage.metrics import structural_similarity, peak_signal_noise_ratio

# --------------------------------------------------
# 1. PATHS AND CONFIGURATION
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

VOLUME_DIR = ROOT / "dataset" / "volumes"
VAL_CSV = ROOT / "val_patients.csv"
CHECKPOINT_PATH = ROOT / "models" / "latest_checkpoint.pth"

OUTPUT_DIR = ROOT / "outputs" / "validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from resunet import ResidualUNet
from normalise import normalise_ct, normalise_mri

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8

print("Device:", DEVICE)

# --------------------------------------------------
# 2. LOAD VALIDATION PATIENT IDS
# --------------------------------------------------

val_df = pd.read_csv(VAL_CSV)

if "patient_id" in val_df.columns:
    patient_ids = val_df["patient_id"].astype(str).tolist()
else:
    patient_ids = val_df.iloc[:, 0].astype(str).tolist()

print("Validation patients:", patient_ids)

# --------------------------------------------------
# 3. LOAD MODEL AND CHECKPOINT
# --------------------------------------------------

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint successfully.")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Training loss:", checkpoint["train_loss"])

# --------------------------------------------------
# 4. INFERENCE ON ONE COMPLETE VOLUME
# --------------------------------------------------

def predict_volume(ct_normalized):

    depth = ct_normalized.shape[2]

    predictions = np.zeros(
        ct_normalized.shape,
        dtype=np.float32
    )

    with torch.no_grad():

        for start in range(0, depth, BATCH_SIZE):

            end = min(start + BATCH_SIZE, depth)

            batch_slices = []

            for z in range(start, end):

                prev_z = max(z - 1, 0)
                next_z = min(z + 1, depth - 1)

                three_channels = np.stack(
                    [
                        ct_normalized[:, :, prev_z],
                        ct_normalized[:, :, z],
                        ct_normalized[:, :, next_z]
                    ],
                    axis=0
                )

                batch_slices.append(three_channels)

            batch = torch.from_numpy(
                np.stack(batch_slices)
            ).float().to(DEVICE)

            if DEVICE.type == "cuda":
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    output = model(batch)
            else:
                output = model(batch)

            output = output.float().cpu().numpy()

            for i, z in enumerate(range(start, end)):
                predictions[:, :, z] = output[i, 0]

    return np.clip(predictions, 0.0, 1.0)


# --------------------------------------------------
# 5. EVALUATE ALL VALIDATION PATIENTS
# --------------------------------------------------

results = []

for patient_id in patient_ids:

    print("\nEvaluating:", patient_id)

    patient_dir = VOLUME_DIR / patient_id

    ct_path = patient_dir / "ct.nii.gz"
    mr_path = patient_dir / "mr.nii.gz"
    mask_path = patient_dir / "mask.nii.gz"

    ct_img = nib.load(str(ct_path))
    mr_img = nib.load(str(mr_path))
    mask_img = nib.load(str(mask_path))

    ct = ct_img.get_fdata().astype(np.float32)
    mr = mr_img.get_fdata().astype(np.float32)
    mask = mask_img.get_fdata().astype(np.uint8)

    assert ct.shape == mr.shape == mask.shape

    # Use the dataset's official normalization functions
    ct_norm = normalise_ct(ct)
    mr_norm = normalise_mri(mr, mask)

    # Predict complete MRI volume
    prediction = predict_volume(ct_norm)

    # Metrics
    full_mae = np.mean(np.abs(prediction - mr_norm))

    mask_bool = mask > 0

    if mask_bool.any():
        masked_mae = np.mean(
            np.abs(prediction[mask_bool] - mr_norm[mask_bool])
        )
    else:
        masked_mae = np.nan

    # Average 2D slice SSIM
    ssim_values = []

    for z in range(mr_norm.shape[2]):
        score = structural_similarity(
            mr_norm[:, :, z],
            prediction[:, :, z],
            data_range=1.0
        )
        ssim_values.append(score)

    mean_ssim = float(np.mean(ssim_values))

    # Full-volume PSNR
    psnr = peak_signal_noise_ratio(
        mr_norm,
        prediction,
        data_range=1.0
    )

    print(f"Full-volume MAE : {full_mae:.6f}")
    print(f"Masked MAE      : {masked_mae:.6f}")
    print(f"Mean slice SSIM : {mean_ssim:.6f}")
    print(f"Full-volume PSNR: {psnr:.4f} dB")

    results.append({
        "patient_id": patient_id,
        "full_volume_mae": full_mae,
        "masked_mae": masked_mae,
        "mean_slice_ssim": mean_ssim,
        "full_volume_psnr": psnr
    })

    # Save predicted MRI using the original CT geometry
    output_img = nib.Nifti1Image(
        prediction.astype(np.float32),
        ct_img.affine,
        ct_img.header
    )

    output_path = OUTPUT_DIR / f"{patient_id}_predicted_mri.nii.gz"

    nib.save(output_img, str(output_path))

    print("Saved:", output_path)

# --------------------------------------------------
# 6. SAVE FINAL RESULTS
# --------------------------------------------------

results_df = pd.DataFrame(results)

results_path = OUTPUT_DIR / "validation_metrics.csv"
results_df.to_csv(results_path, index=False)

print("\n====================================")
print("VALIDATION COMPLETED")
print("====================================")

print(results_df.to_string(index=False))

print("\nAverage metrics:")
print(results_df.drop(columns=["patient_id"]).mean())

print("\nResults saved to:", results_path)


Device: cuda
Validation patients: ['P2658', 'P6123', 'P7363', 'P7480', 'P8947']
Loaded checkpoint successfully.
Checkpoint epoch: 10
Training loss: 0.0491654581887134

Evaluating: P2658
Full-volume MAE : 0.063618
Masked MAE      : 0.106695
Mean slice SSIM : 0.581042
Full-volume PSNR: 18.3026 dB
Saved: D:\PS07_CT2MRI\outputs\validation\P2658_predicted_mri.nii.gz

Evaluating: P6123
Full-volume MAE : 0.074747
Masked MAE      : 0.114213
Mean slice SSIM : 0.556709
Full-volume PSNR: 17.3466 dB
Saved: D:\PS07_CT2MRI\outputs\validation\P6123_predicted_mri.nii.gz

Evaluating: P7363
Full-volume MAE : 0.067603
Masked MAE      : 0.120383
Mean slice SSIM : 0.510815
Full-volume PSNR: 17.8009 dB
Saved: D:\PS07_CT2MRI\outputs\validation\P7363_predicted_mri.nii.gz

Evaluating: P7480
Full-volume MAE : 0.066322
Masked MAE      : 0.112668
Mean slice SSIM : 0.574875
Full-volume PSNR: 17.6242 dB
Saved: D:\PS07_CT2MRI\outputs\validation\P7480_predicted_mri.nii.gz

Evaluating: P8947
Full-volume MAE : 0.060150